# SSB RSS – publiseringer i sammenheng

SSB har tre RSS-feeder som utfyller hverandre:

| Feed | Innhold | Tidsvindu |
|---|---|---|
| `rss/statkal` | statistikkalenderen: planlagte publiseringer | de neste 30 dagene, fra i morgen |
| `rss/` | nyheter: publiserte statistikker og artikler | de siste dagene |
| `rss/statbank` | oppdaterte tabeller i Statistikkbanken | de siste publiseringsdagene |

**Kortnavnet binder feedene sammen.** Hver statistikk i SSB har et kortnavn, for eksempel `kpi`
for Konsumprisindeksen. Det står i feltet `ssbrss:shortname` i alle tre feedene. Med kortnavnet
kan vi derfor koble en planlagt publisering til nyhetssaken og til tabellene som ble oppdatert.

Feedene er vinduer, ikke arkiver: det som faller ut av vinduet, er borte.

Notebooken går gjennom fem steg:

1. hente feedene
2. se på hvordan en post ser ut
3. gjøre hver feed om til en tabell
4. rydde i hver tabell
5. koble tabellene sammen med kortnavnet

In [ ]:
import xml.etree.ElementTree as ET

import pandas as pd
import requests

LANG = "no"  # "en" gir engelske titler
FEEDS = {
    "kalender": "https://www.ssb.no/rss/statkal",
    "nyheter": "https://www.ssb.no/rss/",
    "statbank": "https://www.ssb.no/rss/statbank",
}
MINE = ["kpi", "aku", "konkurs"]  # kortnavn du følger med på
I_DAG = pd.Timestamp.today().normalize()

## 1. Hente feedene

En RSS-feed er en XML-fil på en vanlig nettadresse. Vi henter de tre filene som tekst
og lagrer dem i en ordbok.

In [ ]:
xml = {}
for navn, url in FEEDS.items():
    svar = requests.get(url, params={"lang": LANG}, timeout=30)
    svar.raise_for_status()
    xml[navn] = svar.content.decode("utf-8")
    print(navn, len(xml[navn]), "tegn")

## 2. Slik ser en post ut

Hver post i en feed ligger mellom `<item>` og `</item>`. Her er den første posten i hver feed.

In [ ]:
for navn in FEEDS:
    start = xml[navn].find("<item>")
    slutt = xml[navn].find("</item>") + len("</item>")
    print("-----", navn, "-----")
    print(xml[navn][start:slutt])
    print()

Legg merke til dette:

- `title` og `link` er vanlige RSS-felt. Feltene som begynner med `ssbrss:`, er SSBs egne tillegg.
- `ssbrss:shortname` er kortnavnet. Det finnes i alle tre feedene.
- `ssbrss:date` er publiseringsdatoen.
- `ssbrss:tableid` er tabellnummeret. Det finnes bare i statbank-feeden.
- Titlene er bygd opp ulikt: kalenderen har dato først, og statbank har tabellnummer først.

## 3. Fra XML til tabell

Vi leser de samme feltene fra alle tre feedene, slik at tabellene får like kolonner.
Feltene til SSB ligger i et eget navnerom, og `NS` forteller ElementTree hva `ssbrss:` står for.

In [ ]:
NS = {"ssbrss": "http://www.ssb.no/ns/ssbrss"}


def tekst(item, felt):
    """Teksten i et felt, eller None hvis feltet mangler eller er tomt."""
    verdi = item.findtext(felt, namespaces=NS)
    if verdi is None or verdi.strip() == "":
        return None
    return verdi.strip()


def les_feed(navn):
    """Én rad per post, med de samme kolonnene for alle feedene."""
    rader = []
    for item in ET.fromstring(xml[navn]).iter("item"):
        rader.append({
            "feed": navn,
            "dato": tekst(item, "ssbrss:date"),
            "kortnavn": tekst(item, "ssbrss:shortname"),
            "tabell": tekst(item, "ssbrss:tableid"),
            "tittel": tekst(item, "title"),
            "lenke": tekst(item, "link"),
        })
    df = pd.DataFrame(rader)
    df["dato"] = pd.to_datetime(df["dato"])
    return df


kalender = les_feed("kalender")
nyheter = les_feed("nyheter")
statbank = les_feed("statbank")

In [ ]:
# Hvor mange poster har hver feed, og hvilke datoer dekker den?
for df in [kalender, nyheter, statbank]:
    print(df["feed"][0], len(df), "poster fra", df["dato"].min().date(), "til", df["dato"].max().date())

## 4. Rydde i hver tabell

### Kalenderen: dele tittelen

Tittelen ser slik ut: `22.10.2026: Arbeidskraftundersøkelsen, Tall for september 2026`.
Datoen har vi allerede, så den fjerner vi. Resten deler vi ved det **siste** kommaet,
fordi navnet på statistikken selv kan inneholde komma.

In [ ]:
uten_dato = kalender["tittel"].str.replace(r"^\d{2}\.\d{2}\.\d{4}:\s*", "", regex=True)
deler = uten_dato.str.rsplit(",", n=1, expand=True)
kalender["statistikk"] = deler[0].str.strip()
kalender["periode"] = deler[1].str.strip()
kalender[["dato", "kortnavn", "statistikk", "periode"]].head()

### Statbank: fjerne tabellnummeret fra tittelen

Tittelen ser slik ut: `14549: Avvirkning av …`. Tabellnummeret står allerede i kolonnen `tabell`.

In [ ]:
statbank["tittel"] = statbank["tittel"].str.replace(r"^\d+:\s*", "", regex=True)
statbank[["dato", "kortnavn", "tabell", "tittel"]].head()

### Nyheter: skille publiseringer fra artikler

Nyhetsfeeden inneholder to slags poster:

- **Publiseringer** lenker til statistikksiden, f.eks. `…/statistikk/ledige-stillingar`.
- **Artikler** lenker til en underside, f.eks. `…/statistikk/ledige-stillingar/artikler/nesten-90-000-ledige-stillingar`.

In [ ]:
er_artikkel = nyheter["lenke"].str.contains("/artikler/")
nyheter["type"] = "publisering"
nyheter.loc[er_artikkel, "type"] = "artikkel"
nyheter[["dato", "type", "kortnavn", "tittel"]]

### Nyheter: gi artiklene kortnavn

Artiklene mangler kortnavn, men lenken avslører hvilken statistikk de hører til: delen før
`/artikler/` er adressen til statistikksiden. Publiseringene i nyhetsfeeden og postene i kalenderen
har både den adressen og et kortnavn. Dem bruker vi som oppslagstabell.

In [ ]:
# Oppslagstabell: adressen til statistikksiden -> kortnavn
kjente = pd.concat([nyheter, kalender]).dropna(subset=["kortnavn"]).drop_duplicates("lenke")
side_til_kortnavn = kjente.set_index("lenke")["kortnavn"]

# Slå opp statistikksiden til hver post, og fyll inn der kortnavnet mangler
statistikkside = nyheter["lenke"].str.split("/artikler/").str[0]
nyheter["kortnavn"] = nyheter["kortnavn"].fillna(statistikkside.map(side_til_kortnavn))

# Artikler vi ikke finner noe kortnavn til, får tomt kortnavn
nyheter["kortnavn"] = nyheter["kortnavn"].fillna("")
nyheter[["dato", "type", "kortnavn", "tittel"]]

## 5. Kortnavnet binder feedene sammen

Nå har alle tre tabellene en kolonne `kortnavn`. Tabellen under teller poster per kortnavn
i hver feed, og viser kortnavnene som finnes i mer enn én feed.

In [ ]:
alle = pd.concat([kalender, nyheter, statbank], ignore_index=True)
alle = alle.sort_values(["dato", "kortnavn", "feed"])

antall = pd.crosstab(alle["kortnavn"], alle["feed"])
antall_feeder = (antall > 0).sum(axis=1)
antall[antall_feeder > 1]

Velg ett av kortnavnene over, og se alt de tre feedene sier om den statistikken.

In [ ]:
KORTNAVN = antall_feeder.idxmax()  # eller skriv et kortnavn selv, f.eks. "kpi"
alle.loc[alle["kortnavn"] == KORTNAVN, ["dato", "feed", "kortnavn", "tabell", "tittel"]]

## Én rad per publisering

En publisering er en statistikk som slippes en bestemt dag. Den kjenner vi igjen på
**dato og kortnavn**. Vi lager først én rad per publisering, og kobler så på én opplysning om gangen.

### Steg 1: alle kombinasjoner av dato og kortnavn

In [ ]:
NØKKEL = ["dato", "kortnavn"]

publiseringer = alle[NØKKEL].drop_duplicates()
publiseringer

### Steg 2: perioden tallene gjelder for, fra kalenderen

In [ ]:
perioder = kalender[["dato", "kortnavn", "periode"]].drop_duplicates(NØKKEL)

publiseringer = publiseringer.merge(perioder, on=NØKKEL, how="left")
publiseringer.tail()

### Steg 3: tabellene som ble oppdatert, fra statbank

En publisering oppdaterer ofte flere tabeller. Vi samler tabellnumrene i én tekst per publisering.

In [ ]:
tabeller = statbank[["dato", "kortnavn", "tabell"]].drop_duplicates().sort_values("tabell")
tabeller = tabeller.groupby(NØKKEL)["tabell"].agg(", ".join).reset_index()
tabeller = tabeller.rename(columns={"tabell": "tabeller"})

publiseringer = publiseringer.merge(tabeller, on=NØKKEL, how="left")
publiseringer.head()

### Steg 4: artiklene, fra nyhetsfeeden

In [ ]:
artikler = nyheter[nyheter["type"] == "artikkel"]
artikler = artikler.groupby(NØKKEL)["tittel"].agg(" | ".join).reset_index()
artikler = artikler.rename(columns={"tittel": "artikler"})

publiseringer = publiseringer.merge(artikler, on=NØKKEL, how="left")
publiseringer.head()

### Steg 5: navnet på statistikken og lenken til statistikksiden

Navnet og lenken hører til kortnavnet, ikke til datoen, så her kobler vi bare på `kortnavn`.
Vi henter dem fra publiseringene i nyhetsfeeden og fra kalenderen. Et kortnavn som bare finnes
i statbank-feeden, får ikke noe navn.

In [ ]:
fra_nyheter = nyheter.loc[nyheter["type"] == "publisering", ["kortnavn", "tittel", "lenke"]]
fra_nyheter = fra_nyheter.rename(columns={"tittel": "statistikk"})
fra_kalender = kalender[["kortnavn", "statistikk", "lenke"]]

# Står kortnavnet i begge, beholder vi raden fra nyhetsfeeden, fordi den står først
statistikker = pd.concat([fra_nyheter, fra_kalender]).drop_duplicates("kortnavn")

publiseringer = publiseringer.merge(statistikker, on="kortnavn", how="left")
publiseringer.head()

### Steg 6: er publiseringen planlagt eller publisert?

In [ ]:
publiseringer["status"] = "publisert"
publiseringer.loc[publiseringer["dato"] > I_DAG, "status"] = "planlagt"

publiseringer = publiseringer.sort_values(NØKKEL).reset_index(drop=True)
publiseringer = publiseringer[["dato", "status", "kortnavn", "statistikk", "periode", "tabeller", "artikler", "lenke"]]
publiseringer

## Visninger

Funksjonen `vis` er bare pynt: den gjør statistikknavn og tabellnumre om til klikkbare lenker.
Den har ikke noe med RSS å gjøre, og kan hoppes over.

In [ ]:
def statistikk_lenke(navn, lenke):
    if pd.isna(navn):
        return ""
    return f'<a href="{lenke}">{navn}</a>'


def tabell_lenker(tabeller):
    if pd.isna(tabeller):
        return ""
    lenker = []
    for nummer in tabeller.split(", "):
        lenker.append(f'<a href="https://www.ssb.no/statbank/table/{nummer}">{nummer}</a>')
    return ", ".join(lenker)


def vis(df):
    df = df.copy()
    df["dato"] = df["dato"].dt.strftime("%d.%m.%Y")
    df["statistikk"] = [statistikk_lenke(navn, lenke) for navn, lenke in zip(df["statistikk"], df["lenke"])]
    df["tabeller"] = df["tabeller"].map(tabell_lenker)
    df = df.drop(columns="lenke")
    return df.style.hide(axis="index").format(na_rep="")

In [ ]:
# Publisert de siste dagene
vis(publiseringer[publiseringer["status"] == "publisert"])

In [ ]:
# Planlagt de neste sju dagene
om_en_uke = I_DAG + pd.Timedelta(days=7)
vis(publiseringer[(publiseringer["dato"] > I_DAG) & (publiseringer["dato"] <= om_en_uke)])

In [ ]:
# Mine statistikker
vis(publiseringer[publiseringer["kortnavn"].isin(MINE)])